In [2]:
import pandas as pd
import numpy as np

In [ ]:
# reading the data set header = none is important
sensors = pd.read_csv('../data/raw/secom.data', sep=' ', header=None)

# isolating the labels
labels = pd.read_csv('../data/raw/secom_labels.data', sep=' ', header=None)
labels.columns = ['Result', 'Timestamp']

# axis = 1 stacks them side by side so each sensor reading gets a time and run number
df = pd.concat([labels, sensors], axis=1)

# change -1 pass to 0 and 1 is fail still, following modern ML rules
df['Result'] = df['Result'].replace(-1, 0)
# total shape of the data
print(f"Dataset Shape: {df.shape}")
# 0 is passes and 1 if fails
print(f"Total Passes (0): {df['Result'].value_counts()[0]}")
print(f"Total Fails (1): {df['Result'].value_counts()[1]}")

Dataset Shape: (1567, 592)
Total Passes (0): 1463
Total Fails (1): 104


In [ ]:
# true is cell is empty and false otherwise
# means the the true to find percentage in missing data
missing_pct = df.isnull().mean()
high_missing_cols = missing_pct[missing_pct > 0.50]
print(f"Number of columns missing more than 50% data: {len(high_missing_cols)}")

# find the number of unique values in every column
# columns with all same number get saved
unique_counts = df.nunique()
constant_cols = unique_counts[unique_counts == 1]
print(f"Number of completely constant (zero-variance) columns: {len(constant_cols)}")

Number of columns missing more than 50% data: 28
Number of completely constant (zero-variance) columns: 116


In [5]:
# Combining both lists of data that we want to remove
# set().union() to remove duplicates between lists
cols_to_drop = list(set(high_missing_cols.index).union(set(constant_cols.index)))

# delete from df
df_clean = df.drop(columns=cols_to_drop)

print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {df_clean.shape}")

Original shape: (1567, 592)
Cleaned shape: (1567, 448)


In [9]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler

# separating columns that should not be scaled/imputed with the sensor readings
timestamp_col = df_clean['Timestamp']
result_col = df_clean['Result']
sensor_df = df_clean.drop(columns=['Timestamp', 'Result']).copy()

# scikit-learn expects feature names to all have the same type
sensor_df.columns = sensor_df.columns.astype(str)

# scaling number to a rang between 0-1
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(sensor_df)
df_scaled = pd.DataFrame(scaled_data, columns=sensor_df.columns)

# adjusting K
imputer = KNNImputer(n_neighbors=5)

# fit and transformed data
imputed_array = imputer.fit_transform(df_scaled)

# inverse the transform from scaling
final_imputed_data = scaler.inverse_transform(imputed_array)
df_final = pd.DataFrame(final_imputed_data, columns=sensor_df.columns)

# attach label and timestamp back
df_final['Result'] = result_col.values
df_final['Timestamp'] = timestamp_col.values

In [10]:
# We use .iloc[:, :5] to select all rows, but only the first 5 sensor columns 
# so the output is readable on your screen.

print("--- BEFORE IMPUTATION (Missing Data) ---")
display(sensor_df.iloc[:, :5].describe())

print("\n--- AFTER KNN IMPUTATION (Healed Data) ---")
display(df_final.iloc[:, :5].describe())

--- BEFORE IMPUTATION (Missing Data) ---


,0,1,2,3,4
count,1561.000000,1560.000000,1553.000000,1553.000000,1553.000000
mean,3014.452896,2495.850231,2200.547318,1396.376627,4.197013
std,73.621787,80.407705,29.513152,441.691640,56.355540
min,2743.240000,2158.750000,2060.660000,0.000000,0.681500
25%,2966.260000,2452.247500,2181.044400,1081.875800,1.017700
50%,3011.490000,2499.405000,2201.066700,1285.214400,1.316800
75%,3056.650000,2538.822500,2218.055500,1591.223500,1.525700
max,3356.350000,2846.440000,2315.266700,3715.041700,1114.536600



--- AFTER KNN IMPUTATION (Healed Data) ---


,0,1,2,3,4
count,1567.000000,1567.000000,1567.000000,1567.000000,1567.000000
mean,3014.396841,2495.874170,2200.507365,1397.896804,4.739867
std,73.495693,80.306032,29.443003,441.503427,57.737046
min,2743.240000,2158.750000,2060.660000,0.000000,0.681500
25%,2966.665000,2452.215000,2181.044400,1083.885800,1.017700
50%,3011.320000,2499.460000,2201.066700,1287.353800,1.316800
75%,3056.540000,2539.181000,2218.055500,1591.223500,1.529100
max,3356.350000,2846.440000,2315.266700,3715.041700,1114.536600


In [11]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
# y is the result answer key
# x matrix of sensor readings, test material
y = df_final['Result']
# dropping result because its the answer and timestamp cause its a date (XGBoost cannot handle)
X = df_final.drop(columns=['Result', 'Timestamp'])

# 80 training and 20 testing split
# stratify=y ensures both the 80% and 20% chunks keep that exact 94/6 Pass/Fail ratio
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# using pass and fails to calculate the class weight
# class weight ratio passes/fail
weight_ratio = y_train.value_counts()[0] / y_train.value_counts()[1]
print(f"Calculated scale_pos_weight: {weight_ratio:.2f}\n")

# model for classification
# every time a failed broken wafer catch the error penalty is multiplied by 14
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=weight_ratio, 
    random_state=42, # "random number"
    # confidence based eval metric that punishes more if wrong when more sure
    eval_metric='logloss'
)

# fitting
xgb_model.fit(X_train, y_train)

# predictions from the test portion
predictions = xgb_model.predict(X_test)

print("--- XGBoost Performance ---")
print(classification_report(y_test, predictions))

Calculated scale_pos_weight: 14.10

--- XGBoost Performance ---
              precision    recall  f1-score   support

           0       0.93      1.00      0.96       293
           1       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314



In [15]:
from sklearn.feature_selection import SelectFromModel

# scout model to rank sensor importance
scout_model = xgb.XGBClassifier(
    scale_pos_weight=weight_ratio, 
    random_state=42, 
    eval_metric='logloss'
)
scout_model.fit(X_train, y_train)

# init the dynamic filter
# The threshold='median' tells it to dynamically calculate the median importance score 
# of all sensors, and ONLY keep the sensors that score higher than that median.
dynamic_filter = SelectFromModel(scout_model, threshold='median', prefit=True)

# transform dataset with important sensors
X_train_dynamic = dynamic_filter.transform(X_train)
X_test_dynamic = dynamic_filter.transform(X_test)

print(f"Original sensor count: {X_train.shape[1]}")
print(f"Dynamically selected sensor count: {X_train_dynamic.shape[1]}\n")

# training model on ranked data
xgb_model_dynamic = xgb.XGBClassifier(
    scale_pos_weight=weight_ratio, 
    random_state=42, 
    eval_metric='logloss'
)
xgb_model_dynamic.fit(X_train_dynamic, y_train)

predictions_dynamic = xgb_model_dynamic.predict(X_test_dynamic)

print("--- Dynamic XGBoost Performance ---")
print(classification_report(y_test, predictions_dynamic))

c:\Users\zraym\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\zraym\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Original sensor count: 446
Dynamically selected sensor count: 223

--- Dynamic XGBoost Performance ---
              precision    recall  f1-score   support

           0       0.93      1.00      0.96       293
           1       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314



In [28]:
from sklearn.feature_selection import SelectKBest, f_classif
import numpy as np

# filter only the top 15 sensors from teh data set
elite_filter = SelectKBest(score_func=f_classif, k=10)

X_train_elite = elite_filter.fit_transform(X_train, y_train)
X_test_elite = elite_filter.transform(X_test)

# listing out the top 15
selected_indices = elite_filter.get_support(indices=True)
print(f"The 10 elite sensors chosen by math: {selected_indices}\n")

# retrain on these 15 sensors
xgb_model_elite = xgb.XGBClassifier(
    scale_pos_weight=weight_ratio, 
    random_state=42, 
    eval_metric='logloss',
    max_depth=4 # shallow trees for less noise
)

xgb_model_elite.fit(X_train_elite, y_train)
predictions_elite = xgb_model_elite.predict(X_test_elite)

print("--- Elite 10-Sensor XGBoost Performance ---")
print(classification_report(y_test, predictions_elite))

The 10 elite sensors chosen by math: [ 19  54  59  93 112 116 277 330 331 388]

--- Elite 10-Sensor XGBoost Performance ---
              precision    recall  f1-score   support

           0       0.94      0.97      0.96       293
           1       0.27      0.14      0.19        21

    accuracy                           0.92       314
   macro avg       0.61      0.56      0.57       314
weighted avg       0.90      0.92      0.90       314



In [29]:
# 1. Ask the AI for its exact probability scores instead of a hard Pass/Fail
# The [:, 1] extracts the percentage chance of a Fail (Class 1)
probabilities = xgb_model_elite.predict_proba(X_test_elite)[:, 1]

# 2. Turn the "Paranoia Dial" way up.
# If the AI is even 15% sure the machine is breaking, halt the factory.
custom_threshold = 0.15

# 3. Apply the custom threshold mathematically
# This converts the probabilities into hard 1s and 0s based on our new rule
custom_predictions = (probabilities >= custom_threshold).astype(int)

# 4. Print the new reality
print(f"--- Paranoia Threshold ({custom_threshold * 100}%) Performance ---")
print(classification_report(y_test, custom_predictions))

--- Paranoia Threshold (15.0%) Performance ---
              precision    recall  f1-score   support

           0       0.95      0.91      0.93       293
           1       0.21      0.33      0.26        21

    accuracy                           0.87       314
   macro avg       0.58      0.62      0.59       314
weighted avg       0.90      0.87      0.89       314



In [30]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import KNNImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

# =====================================================================
# 1. CLEAN SPLIT FIRST (Zero Leakage)
# =====================================================================
# Separate target and features immediately
y = df_clean['Result']
X = df_clean.drop(columns=['Timestamp', 'Result'])
X.columns = X.columns.astype(str)

# First split: Separate out the final Test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: Split the remaining 80% into Train (60%) and Validation (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Train shapes: {X_train.shape}, Val shapes: {X_val.shape}, Test shapes: {X_test.shape}\n")

# =====================================================================
# 2. FIT PREPROCESSING ON TRAINING ONLY
# =====================================================================
# Scale
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Impute
imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train_scaled)
X_val_imputed = imputer.transform(X_val_scaled)
X_test_imputed = imputer.transform(X_test_scaled)

# Feature Selection (Elite 10 Sensors)
selector = SelectKBest(score_func=f_classif, k=10)
X_train_final = selector.fit_transform(X_train_imputed, y_train)
X_val_final = selector.transform(X_val_imputed)
X_test_final = selector.transform(X_test_imputed)

# =====================================================================
# 3. TRAIN CONTROLLED XGBOOST MODEL
# =====================================================================
weight_ratio = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.5,
    reg_lambda=3,
    scale_pos_weight=weight_ratio, # Tuning this between [3, 5, 8, 10, 14, 18] is a great next step
    random_state=42,
    eval_metric="aucpr",  # Optimizing for Area Under Precision-Recall Curve
    tree_method="hist"
)

xgb_model.fit(X_train_final, y_train)

# =====================================================================
# 4. TUNING THRESHOLD ON VALIDATION SET
# =====================================================================
val_probabilities = xgb_model.predict_proba(X_val_final)[:, 1]

# We evaluate threshold performance completely isolated from the test set
chosen_threshold = 0.18 
val_predictions = (val_probabilities >= chosen_threshold).astype(int)

print("--- VALIDATION SET PERFORMANCE (For Threshold Selection) ---")
print(classification_report(y_val, val_predictions))

# =====================================================================
# 5. FINAL EVALUATION ON TEST SET (The Truth)
# =====================================================================
test_probabilities = xgb_model.predict_proba(X_test_final)[:, 1]
final_predictions = (test_probabilities >= chosen_threshold).astype(int)

print("\n--- FINAL TEST SET PERFORMANCE ---")
print(classification_report(y_test, final_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_test, final_predictions))
print(f"Average Precision Score: {average_precision_score(y_test, test_probabilities):.4f}")

Train shapes: (939, 446), Val shapes: (314, 446), Test shapes: (314, 446)

--- VALIDATION SET PERFORMANCE (For Threshold Selection) ---
              precision    recall  f1-score   support

           0       0.95      0.61      0.74       293
           1       0.10      0.57      0.16        21

    accuracy                           0.61       314
   macro avg       0.52      0.59      0.45       314
weighted avg       0.89      0.61      0.71       314


--- FINAL TEST SET PERFORMANCE ---
              precision    recall  f1-score   support

           0       0.97      0.67      0.79       293
           1       0.13      0.71      0.22        21

    accuracy                           0.67       314
   macro avg       0.55      0.69      0.51       314
weighted avg       0.91      0.67      0.75       314

Confusion Matrix:
[[195  98]
 [  6  15]]
Average Precision Score: 0.2259


c:\Users\zraym\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [ 66 180 183 273 276 368] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\zraym\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [ ]:
from sklearn.model_selection import GridSearchCV

# 1. Define the "Grid" of parameters we want to test
# The computer will test every possible combination of these numbers
param_grid = {
    'max_depth': [3, 4, 5],                  # How complex the logic chain gets
    'learning_rate': [0.01, 0.05, 0.1],      # How fast it learns from mistakes
    'scale_pos_weight': [8, 10, 14, 18],     # The penalty dial for missing a failure
    'n_estimators': [200, 300]               # How many total trees to build
}

# 2. Initialize a fresh, empty XGBoost model
base_xgb = xgb.XGBClassifier(
    random_state=42, 
    eval_metric="aucpr",
    tree_method="hist",
    subsample=0.8,         # Keeping regularization stable to prevent overfitting
    colsample_bytree=0.8
)

# 3. Set up the Automated Grid Search
# cv=5 means it will cross-validate the training data 5 times for EVERY combination
grid_search = GridSearchCV(
    estimator=base_xgb,
    param_grid=param_grid,
    scoring='average_precision', # CRITICAL: Forces it to optimize for the rare failures
    cv=5,
    verbose=1, # Prints out the progress so you know it hasn't frozen
    n_jobs=-1  # Uses all your computer's CPU cores to do the math faster
)

print("Starting the automated grid search... This may take a minute.\n")

# 4. Release the machine to crunch the math
grid_search.fit(X_train_final, y_train)

# 5. Extract the absolute best model
champion_model = grid_search.best_estimator_

print("--- TUNING COMPLETE ---")
print(f"The Champion Parameters: {grid_search.best_params_}\n")

# =====================================================================
# 6. EVALUATE THE CHAMPION ON THE VALIDATION SET
# =====================================================================
val_probabilities = champion_model.predict_proba(X_val_final)[:, 1]

# You can still tweak this threshold manually if you want to push recall higher
chosen_threshold = 0.20 
val_predictions = (val_probabilities >= chosen_threshold).astype(int)

print("--- CHAMPION MODEL: VALIDATION PERFORMANCE ---")
print(classification_report(y_val, val_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_val, val_predictions))

Starting the automated grid search... This may take a minute.

Fitting 5 folds for each of 72 candidates, totalling 360 fits
--- TUNING COMPLETE ---
The Champion Parameters: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'scale_pos_weight': 8}

--- CHAMPION MODEL: VALIDATION PERFORMANCE ---
              precision    recall  f1-score   support

           0       0.95      0.49      0.65       293
           1       0.08      0.62      0.14        21

    accuracy                           0.50       314
   macro avg       0.51      0.56      0.39       314
weighted avg       0.89      0.50      0.61       314

Confusion Matrix:
[[144 149]
 [  8  13]]


In [32]:
from sklearn.metrics import precision_recall_curve

# 1. Get the probabilities from the champion model on the Validation set
val_probabilities = champion_model.predict_proba(X_val_final)[:, 1]

# 2. Calculate precision, recall, and thresholds across the entire curve
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probabilities)

# 3. Calculate the F1 score for every single threshold
# (Adding a tiny number 1e-8 prevents dividing by zero)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)

# 4. Find the index of the highest F1 score
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"The mathematically optimal threshold is: {optimal_threshold:.4f}\n")

# 5. Apply the perfect threshold to the Validation Set
optimal_val_predictions = (val_probabilities >= optimal_threshold).astype(int)

print("--- CHAMPION MODEL: OPTIMIZED VALIDATION PERFORMANCE ---")
print(classification_report(y_val, optimal_val_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_val, optimal_val_predictions))

The mathematically optimal threshold is: 0.3612

--- CHAMPION MODEL: OPTIMIZED VALIDATION PERFORMANCE ---
              precision    recall  f1-score   support

           0       0.96      0.74      0.84       293
           1       0.13      0.52      0.21        21

    accuracy                           0.73       314
   macro avg       0.54      0.63      0.52       314
weighted avg       0.90      0.73      0.79       314

Confusion Matrix:
[[218  75]
 [ 10  11]]


In [33]:
from sklearn.pipeline import Pipeline

# 1. Fuse the Selector and the AI into a single "Pipeline" object
full_pipeline = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('classifier', xgb.XGBClassifier(
        random_state=42, 
        eval_metric="aucpr",
        tree_method="hist",
        n_estimators=200,      # Locking these down to save compute time
        subsample=0.8,
        colsample_bytree=0.8
    ))
])

# 2. Set up the Advanced Grid
# Notice the double underscores (__) connecting the step name to the parameter
advanced_param_grid = {
    'feature_selection__k': [5, 10, 20, 35],             # Testing different sensor counts
    'classifier__max_depth': [3, 4],                     # Shallow trees
    'classifier__learning_rate': [0.01, 0.05],
    'classifier__scale_pos_weight': [10, 14, 18]         # Pushing the penalty back up
}

# 3. Run the Advanced Search
advanced_grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=advanced_param_grid,
    scoring='average_precision', 
    cv=5,
    verbose=1, 
    n_jobs=-1  
)

print("Starting the Advanced Pipeline Search...\n")
advanced_grid_search.fit(X_train_imputed, y_train)

# 4. Extract and Evaluate the New Champion
print("--- ADVANCED TUNING COMPLETE ---")
print(f"Optimal Parameters: {advanced_grid_search.best_params_}\n")

best_pipeline = advanced_grid_search.best_estimator_
val_probabilities = best_pipeline.predict_proba(X_val_imputed)[:, 1]

# Re-run the automatic threshold calculation
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probabilities)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
optimal_threshold = thresholds[np.argmax(f1_scores)]

print(f"New Optimal Threshold: {optimal_threshold:.4f}\n")

optimal_val_predictions = (val_probabilities >= optimal_threshold).astype(int)

print("--- NEW CHAMPION PIPELINE: VALIDATION PERFORMANCE ---")
print(classification_report(y_val, optimal_val_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_val, optimal_val_predictions))

Starting the Advanced Pipeline Search...

Fitting 5 folds for each of 48 candidates, totalling 240 fits
--- ADVANCED TUNING COMPLETE ---
Optimal Parameters: {'classifier__learning_rate': 0.01, 'classifier__max_depth': 4, 'classifier__scale_pos_weight': 18, 'feature_selection__k': 10}

New Optimal Threshold: 0.4299

--- NEW CHAMPION PIPELINE: VALIDATION PERFORMANCE ---
              precision    recall  f1-score   support

           0       0.95      0.71      0.81       293
           1       0.11      0.52      0.19        21

    accuracy                           0.69       314
   macro avg       0.53      0.62      0.50       314
weighted avg       0.90      0.69      0.77       314

Confusion Matrix:
[[207  86]
 [ 10  11]]


c:\Users\zraym\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [ 66 180 183 273 276 368] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\zraym\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import KNNImputer
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, average_precision_score

# =====================================================================
# 1. THE 3-WAY SPLIT (Strict Isolation)
# =====================================================================
X = df_clean.drop(columns=['Timestamp', 'Result'])
y = df_clean['Result']
X.columns = X.columns.astype(str)

# Split 1: Isolate the ultimate Test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

# Split 2: Isolate Validation (15%) from the remaining Train (70%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=42
)

print(f"Shapes -> Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}\n")

# =====================================================================
# 2. THE ZERO-LEAKAGE PIPELINE
# =====================================================================
# We define the steps, but we DO NOT fit them yet. The GridSearch will handle it.
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('imputer', KNNImputer(n_neighbors=5)),
    ('selector', 'passthrough'), # Placeholder to swap selectors dynamically
    ('model', xgb.XGBClassifier(
        random_state=42, 
        eval_metric='aucpr', 
        tree_method='hist',
        n_estimators=200,
        subsample=0.8,
        colsample_bytree=0.8
    ))
])

# =====================================================================
# 3 & 4 & 5. THE MASTER GRID SEARCH
# =====================================================================
# We define two separate grids: one testing SelectKBest, one testing SelectFromModel
param_grid = [
    {
        'selector': [SelectKBest(score_func=f_classif)],
        'selector__k': [10, 20, 50, 100],
        'model__scale_pos_weight': [5, 10, 14, 18],
        'model__max_depth': [3, 4]
    },
    {
        # Using a basic Random Forest to scout features catches interactions that KBest misses
        'selector': [SelectFromModel(xgb.XGBClassifier(random_state=42, max_depth=3))],
        'selector__max_features': [10, 20, 50, 100],
        'selector__threshold': [-np.inf], # Forces it to pick exactly 'max_features' count
        'model__scale_pos_weight': [5, 10, 14, 18],
        'model__max_depth': [3, 4]
    }
]

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='average_precision', # Tuning purely for the rare failures
    cv=3, # 3 folds speeds up the heavy computation
    verbose=1,
    n_jobs=-1
)

print("Starting Master Grid Search...")
grid_search.fit(X_train, y_train)

champion_pipe = grid_search.best_estimator_
print("\n--- CHAMPION PARAMETERS ---")
print(grid_search.best_params_)

# =====================================================================
# FIXING THE SENSOR ID BUG
# =====================================================================
# Extract the active selector from the winning pipeline
winning_selector = champion_pipe.named_steps['selector']
# Map the boolean mask back to the actual string names of the training columns
selected_sensor_names = X_train.columns[winning_selector.get_support()]
print(f"\nTrue Physical Sensors Kept: {selected_sensor_names.tolist()}\n")

# =====================================================================
# 6. TUNE THRESHOLD ON VALIDATION SET
# =====================================================================
val_probs = champion_pipe.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)

# Calculate F1 for all thresholds and find the peak
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"Validation-Optimized Threshold: {optimal_threshold:.4f}\n")

# =====================================================================
# 7. THE FINAL, HONEST TEST SCORE
# =====================================================================
test_probs = champion_pipe.predict_proba(X_test)[:, 1]
final_test_predictions = (test_probs >= optimal_threshold).astype(int)

print("=== TRUE PRODUCTION PERFORMANCE (TEST SET) ===")
print(classification_report(y_test, final_test_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_test, final_test_predictions))
print(f"Average Precision (AUCPR): {average_precision_score(y_test, test_probs):.4f}")

Shapes -> Train: (1096, 446), Val: (235, 446), Test: (236, 446)

Starting Master Grid Search...
Fitting 3 folds for each of 64 candidates, totalling 192 fits


c:\Users\zraym\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_validation.py:489: FitFailedWarning: 
96 fits failed out of a total of 192.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
96 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\zraym\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\zraym\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\zraym\AppDa


--- CHAMPION PARAMETERS ---
{'model__max_depth': 3, 'model__scale_pos_weight': 18, 'selector': SelectKBest(), 'selector__k': 100}

True Physical Sensors Kept: ['10', '14', '21', '22', '26', '28', '33', '37', '56', '58', '59', '64', '68', '70', '78', '79', '81', '90', '95', '100', '103', '114', '115', '121', '122', '123', '124', '125', '126', '127', '129', '130', '133', '135', '145', '159', '160', '163', '164', '165', '170', '172', '174', '175', '180', '183', '195', '196', '197', '199', '200', '203', '210', '213', '222', '249', '270', '280', '294', '295', '298', '299', '300', '310', '316', '319', '331', '337', '348', '351', '360', '387', '408', '430', '431', '434', '435', '436', '441', '443', '445', '446', '452', '455', '460', '467', '468', '469', '471', '477', '494', '510', '521', '550', '551', '554', '557', '562', '587', '588']

Validation-Optimized Threshold: 0.0469

=== TRUE PRODUCTION PERFORMANCE (TEST SET) ===
              precision    recall  f1-score   support

           0   